In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")
tags = pd.read_csv("tags.csv")
links = pd.read_csv("links.csv")

In [3]:
print("movies:", movies.shape)
print("ratings:", ratings.shape)
print("tags:", tags.shape)
print("links:", links.shape)

movies: (87585, 3)
ratings: (32000204, 4)
tags: (2000072, 4)
links: (87585, 3)


In [4]:
#选择一个用户，看评分过多少电影
user_id = 1

user_ratings = ratings[ratings["userId"] == user_id].copy()

print("User:", user_id)
print("Number of rated movies:", len(user_ratings))

User: 1
Number of rated movies: 141


In [5]:
#把电影信息的表合并起来
user_ratings = user_ratings.merge(
    movies,
    on="movieId",
    how="left"
)
user_ratings[
    ["title", "genres", "rating"]
].sort_values(
    "rating",
    ascending=False
).head(20)

,title,genres,rating
45,Platoon (1986),Drama|War,5.0
31,North by Northwest (1959),Action|Adventure|Mystery|Romance|Thriller,5.0
34,Sabrina (1954),Comedy|Romance,5.0
101,Dangerous Liaisons (1988),Drama|Romance,5.0
36,Citizen Kane (1941),Drama|Mystery,5.0
37,All About Eve (1950),Drama,5.0
38,"Women, The (1939)",Comedy,5.0
39,To Catch a Thief (1955),Crime|Mystery|Romance|Thriller,5.0
76,Patton (1970),Drama|War,5.0
41,Secrets & Lies (1996),Drama,5.0


In [6]:
#找出用户1喜欢的电影类型
##4分以上的电影
liked_movies = user_ratings[
    user_ratings["rating"] >= 4
].copy()
print("Liked movies:", len(liked_movies))

Liked movies: 83


In [7]:
#把不同的genre拆开
genre_list = []

for genres in liked_movies["genres"].dropna():
    genre_list.extend(genres.split("|"))

In [8]:
genre_counts = pd.Series(genre_list).value_counts()

genre_counts

Drama        63
Comedy       28
Romance      20
Action       14
War          12
Sci-Fi       11
Adventure    11
Crime        10
Thriller      9
Mystery       6
Fantasy       2
Children      1
Western       1
Horror        1
Film-Noir     1
Name: count, dtype: int64

In [9]:
#每个 Genre 对应电影的平均评分
genre_scores = {}

for genre in genre_counts.index:
    
    genre_movies = liked_movies[
        liked_movies["genres"].fillna("").str.contains(
            genre,
            regex=False
        )
    ]
    
    genre_scores[genre] = genre_movies["rating"].mean()

In [10]:
genre_scores = pd.Series(genre_scores).sort_values(
    ascending=False
)

genre_scores

Crime        5.000000
Children     5.000000
Horror       5.000000
Film-Noir    5.000000
Sci-Fi       4.909091
Mystery      4.833333
Thriller     4.777778
Action       4.714286
Drama        4.650794
Adventure    4.636364
Comedy       4.607143
Romance      4.600000
War          4.583333
Fantasy      4.500000
Western      4.000000
dtype: float64

In [13]:
#找出用户偏好的genre
genre_preference = pd.DataFrame({
    "count": genre_counts,
    "avg_rating": genre_scores
})

genre_preference["score"] = (
    genre_preference["count"] *
    genre_preference["avg_rating"]
)

genre_preference = genre_preference.sort_values(
    "score",
    ascending=False
)

genre_preference

genre_preference.head(10)

,count,avg_rating,score
Drama,63,4.650794,293.0
Comedy,28,4.607143,129.0
Romance,20,4.600000,92.0
Action,14,4.714286,66.0
War,12,4.583333,55.0
Sci-Fi,11,4.909091,54.0
Adventure,11,4.636364,51.0
Crime,10,5.000000,50.0
Thriller,9,4.777778,43.0
Mystery,6,4.833333,29.0


In [14]:
#定义用户最喜欢的genre
favorite_genres = genre_preference.head(5).index.tolist()

favorite_genres

['Drama', 'Comedy', 'Romance', 'Action', 'War']

In [15]:
#找出用户没看过的电影
watched_movies = set(
    user_ratings["movieId"]
)
candidate_movies = movies[
    ~movies["movieId"].isin(watched_movies)
].copy()
print("Total movies:", len(movies))
print("Watched movies:", len(watched_movies))
print("Candidate movies:", len(candidate_movies))

Total movies: 87585
Watched movies: 141
Candidate movies: 87444


In [19]:
#给候选电影算分
def genre_score(genres):
    if pd.isna(genres):
        return 0
    
    movie_genres = set(genres.split("|"))
    
    return len(
        movie_genres.intersection(favorite_genres)
    )
candidate_movies["genre_score"] = (
    candidate_movies["genres"]
    .apply(genre_score)
) 
candidate_movies["genre_score"].value_counts().sort_index()
#去掉等于0的电影
candidate_movies = candidate_movies[
    candidate_movies["genre_score"] > 0
].copy()

In [20]:
#计算每部电影评分
movie_stats = ratings.groupby("movieId").agg(
    avg_rating=("rating", "mean"),
    rating_count=("rating", "count")
).reset_index()
movie_stats.head()

,movieId,avg_rating,rating_count
0,1,3.897438,68997
1,2,3.275758,28904
2,3,3.139447,13134
3,4,2.845331,2806
4,5,3.059602,13154


In [22]:
recommendations = candidate_movies.merge(
    movie_stats,
    on="movieId",
    how="left"
)
print(recommendations.shape)
print(recommendations.columns.tolist())
recommendations.head()

(57810, 6)
['movieId', 'title', 'genres', 'genre_score', 'avg_rating', 'rating_count']


,movieId,title,genres,genre_score,avg_rating,rating_count
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1,3.897438,68997.0
1,3,Grumpier Old Men (1995),Comedy|Romance,2,3.139447,13134.0
2,4,Waiting to Exhale (1995),Comedy|Drama|Romance,3,2.845331,2806.0
3,5,Father of the Bride Part II (1995),Comedy,1,3.059602,13154.0
4,6,Heat (1995),Action|Crime|Thriller,1,3.868277,29490.0


In [23]:
#丢掉没有评分的电影
recommendations = recommendations.dropna(
    subset=["avg_rating", "rating_count"]
)

In [24]:
#计算所有电影平均分
global_mean = movie_stats["avg_rating"].mean()

C = 50

recommendations["weighted_rating"] = (
    recommendations["rating_count"] /
    (recommendations["rating_count"] + C)
    * recommendations["avg_rating"]
    +
    C /
    (recommendations["rating_count"] + C)
    * global_mean
)

In [26]:
#用户喜欢程度60%+电影本身质量40%
##（最终得分是一个简单的启发式排序函数，而不是一个经过学习训练的推荐模型。）
recommendations["final_score"] = (
    0.6 * recommendations["genre_score"]
    +
    0.4 * recommendations["weighted_rating"]
)

In [27]:
recommendations = recommendations.sort_values(
    "final_score",
    ascending=False
)

In [28]:
#推荐前二十
top20 = recommendations[
    [
        "title",
        "genres",
        "genre_score",
        "avg_rating",
        "rating_count",
        "weighted_rating",
        "final_score"
    ]
].head(20)

top20

,title,genres,genre_score,avg_rating,rating_count,weighted_rating,final_score
7057,"White Sun of the Desert, The (Beloe solntse pu...",Action|Adventure|Comedy|Drama|Romance|War,5,3.738220,191.0,3.586117,4.434447
13659,Lazybones (1925),Action|Comedy|Drama|Romance|War,5,2.500000,8.0,2.935416,4.174166
964,Henry V (1989),Action|Drama|Romance|War,4,4.100955,4819.0,4.089701,4.035880
2073,Spartacus (1960),Action|Drama|Romance|War,4,3.826786,6033.0,3.820032,3.928013
8510,Life is a Miracle (Zivot je cudo) (2004),Comedy|Drama|Musical|Romance|War,4,3.844311,167.0,3.650941,3.860376
3872,"Stunt Man, The (1980)",Action|Adventure|Comedy|Drama|Romance|Thriller,4,3.693017,759.0,3.650500,3.860200
1485,Wings (1927),Action|Drama|Romance|War,4,3.684864,403.0,3.609832,3.843933
3753,Operation Petticoat (1959),Action|Comedy|Romance|War,4,3.637399,746.0,3.597681,3.839072
7482,Revolutionary Girl Utena: Adolescence of Utena...,Action|Adventure|Animation|Comedy|Drama|Fantas...,4,3.983051,59.0,3.534441,3.813777
120,Rob Roy (1995),Action|Drama|Romance|War,4,3.529142,12868.0,3.527114,3.810845
